# NGLab Tutorial #1: Quickstart Introduction

Welcome to **Nothing Gambles Like A Bot (NGLab)**—a high-performance financial intelligence platform that bridges Rust simulation engines, Python deep learning, and TypeScript visualization.

## Learning Objectives

By the end of this notebook, you will understand:
1. The core philosophy behind NGLab's design
2. How Rust, Python, and TypeScript components communicate
3. The high-level architecture and data flow

---

## 1. The NGLab Philosophy

Financial markets are **adversarial**, **multimodal**, and fundamentally **non-stationary**. NGLab was built on three core pillars:

### 1.1 Zero-Latency Simulation

In Reinforcement Learning, the "Step" is the atom of time. Speed matters:

| Step Latency | Training Time (1M steps) |
|--------------|-------------------------|
| 100ms (Python) | 27 hours |
| 100μs (Rust) | **1.6 minutes** |

Rust's strict memory management and lack of GC enable deterministic, microsecond-level simulations.

### 1.2 Multi-Scale Modeling

We fuse three scales of data:
1. **Microstructure**: Order book depth, imbalance (Source: Rust CLOB)
2. **Price Action**: OHLCV candlesticks, technical indicators (Source: Project Moon)
3. **Global Alpha**: Sentiment clusters, news flow (Source: Python Deep Learning)

### 1.3 Real-Time Observability

The GUI shows:
- Live bid-ask ladder
- Real-time risk scores (VaR)
- Transformer attention maps

In [ ]:
# Let's verify our environment setup
import sys
import numpy as np
import torch

print(f"Python version: {sys.version}")
print(f"NumPy version: {np.__version__}")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

## 2. High-Level Architecture

NGLab uses a **Hybrid Polyglot Architecture**. Each language is chosen for its strengths:

```
┌─────────────────────────────────────────────────────────────┐
│                     TypeScript GUI                          │
│  (React 19 + Tauri 2.0 + Lightweight Charts)                │
│             "The Observation Layer"                         │
└────────────────────┬────────────────────────────────────────┘
                     │ Tauri IPC (JSON Events)
                     ▼
┌─────────────────────────────────────────────────────────────┐
│                   Rust Core Engine                          │
│  (OrderBook + TradingEnv + Risk Management)                 │
│               "The Physics"                                 │
└────────────────────┬────────────────────────────────────────┘
                     │ PyO3 Bridge (Zero-Copy)
                     ▼
┌─────────────────────────────────────────────────────────────┐
│                 Python Pipeline                             │
│  (Deep Learning + RL + Hyperparameter Optimization)         │
│            "The Decision Engine"                            │
└─────────────────────────────────────────────────────────────┘
```

### 2.1 Component Breakdown

| Component | Language | Responsibility |
|-----------|----------|----------------|
| **Rust Core** | Rust | High-frequency simulation, order matching, risk management |
| **Python Pipeline** | Python | Model training, research, hyperparameter optimization |
| **TypeScript GUI** | TypeScript | Real-time visualization, analytics, user interaction |

### 2.2 Communication Protocols

#### Rust ⇄ Python (PyO3)

We use the `#[pyclass]` attribute on Rust structs to expose them as Python classes:

In [ ]:
# Try importing the Rust-backed nglab module
try:
    import nglab
    print("✓ Successfully imported nglab Rust module")
    print(f"Available classes: {[x for x in dir(nglab) if not x.startswith('_')]}")
except ImportError as e:
    print(f"⚠ nglab module not found: {e}")
    print("You may need to build the Rust extension:")
    print("  cd rust && maturin develop")

#### Rust ⇄ TypeScript (Tauri IPC)

The Rust backend emits JSON-serialized events:

```rust
app.emit_all("arena-update", StepInfo {
    step: step_count,
    portfolio_value: env.portfolio_value,
    position: env.position,
});
```

TypeScript listeners receive these events in real-time.

#### Python ⇄ Configuration (Hydra)

All hyperparameters are centralized:

In [ ]:
# Example: Loading Hydra configuration
from pathlib import Path
import yaml

# Check if config directory exists
config_path = Path("../python/src/conf")
if config_path.exists():
    print(f"✓ Config directory found: {config_path.absolute()}")
    
    # List available configurations
    config_files = list(config_path.rglob("*.yaml"))
    print(f"\nFound {len(config_files)} config files:")
    for cfg in config_files[:5]:  # Show first 5
        print(f"  - {cfg.relative_to(config_path)}")
else:
    print(f"⚠ Config directory not found at {config_path}")

## 3. Data Flow Example

Let's trace how a single trading decision flows through the system:

1. **Rust OrderBook** receives market data (BTC price update)
2. **Rust TradingEnv** generates observation (zero-copy to Python)
3. **Python Agent** (TSMamba model) predicts action: `[0.2, 0.8]` → BUY
4. **Rust OrderBook** executes order, calculates slippage
5. **Rust Portfolio** updates cash/position
6. **Tauri IPC** emits event to TypeScript
7. **React GUI** updates chart, shows new position

**Total latency**: ~150μs (Rust) + 2ms (Python inference) + 16ms (GUI frame) = **<20ms**

## 4. Quick Architecture Validation

In [ ]:
# Validate key imports
components_status = {}

# Check Rust bindings
try:
    import nglab
    components_status['Rust (PyO3)'] = '✓ Available'
except ImportError:
    components_status['Rust (PyO3)'] = '✗ Not built (run: maturin develop)'

# Check Python ML stack
try:
    import torch
    import lightning as L
    components_status['PyTorch + Lightning'] = '✓ Available'
except ImportError as e:
    components_status['PyTorch + Lightning'] = f'✗ Missing: {e}'

# Check RL libraries
try:
    import gymnasium
    components_status['Gymnasium'] = '✓ Available'
except ImportError:
    components_status['Gymnasium'] = '✗ Not installed'

# Check config management
try:
    import hydra
    from omegaconf import OmegaConf
    components_status['Hydra'] = '✓ Available'
except ImportError:
    components_status['Hydra'] = '✗ Not installed'

# Display results
print("\n=== NGLab Component Status ===")
for component, status in components_status.items():
    print(f"{component:.<35} {status}")

## Summary

In this notebook, you learned:

✅ NGLab's three core pillars: Zero-Latency, Multi-Scale Modeling, Real-Time Observability  
✅ The hybrid polyglot architecture (Rust + Python + TypeScript)  
✅ How components communicate via PyO3, Tauri IPC, and Hydra  
✅ The data flow for a single trading decision  

## Next Steps

Continue to **Notebook #2**: Rust OrderBook Basics to dive into the order matching engine!

---

*For questions or issues, check the main [TUTORIAL.md](/TUTORIAL.md) or [ARCHITECTURE.md](/ARCHITECTURE.md)*